# RQ3: Meta-Learning Best Generator Prediction

**Research Question 3 (RQ3):** *Can a meta-model predict the best generator from dataset meta-features alone?*

This notebook reports the leave-one-dataset-out (LODO) evaluation of the RandomForest meta-predictor
trained on OpenML-CC18 dataset profiles (`general`, `statistical`, and `info-theory` meta-features via `pymfe`)
to predict the generator that achieves the highest balanced fidelity-utility-privacy score among candidates
that pass the fail-closed privacy gate.

> **Note:** RQ3 is framed as **exploratory**. With ~25 benchmark datasets, a meta-model that does not beat
> the majority baseline is a valid (negative) result and is reported as such. Nothing here is tuned on the
> held-out dataset.

Inputs (read only): `meta/lodo.json` and `meta/predictor.joblib` / `meta/predictor.json` from the artifact
store (written by `experiments/run.py meta-eval [--train]`), the per-dataset runs `meta-cc18_<id>` and
meta-features in the same store, `experiments/conf/meta/datasets.yaml`, and
`outputs/meta_logs/mst_bounds.txt` (written by `scripts/run_meta_benchmark.sh`). Every section degrades to an
explanation when the data are insufficient (e.g. the 2-dataset smoke run).

Run headless (meta env):
`bash scripts/envrun.sh meta python -m jupyter nbconvert --to notebook --execute --output-dir outputs/notebooks --output rq3_meta_executed.ipynb notebooks/rq3_meta.ipynb`

In [ ]:
# Configuration -- edit here, or set STL_STORAGE_ROOT (local path or fsspec URL) before starting Jupyter.
import os
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "experiments" / "conf" / "meta" / "datasets.yaml").exists())
STORE_ROOT = os.environ.get("STL_STORAGE_ROOT") or str(REPO / "outputs" / "store")
if "://" not in STORE_ROOT and not Path(STORE_ROOT).is_absolute():
    STORE_ROOT = str(REPO / STORE_ROOT)  # relative roots are relative to the repo, like the CLI
DATASETS_YAML = REPO / "experiments" / "conf" / "meta" / "datasets.yaml"
MST_BOUNDS = REPO / "outputs" / "meta_logs" / "mst_bounds.txt"
RUN_PREFIX = "meta-"   # same as `meta-eval --run-prefix`
TOP_N = 15             # feature importances shown

print(f"repo:          {REPO}")
print(f"store:         {STORE_ROOT}")
print(f"datasets yaml: {DATASETS_YAML}")
print(f"mst bounds:    {MST_BOUNDS} (exists: {MST_BOUNDS.exists()})")

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from stl_meta import meta
from stl_runner import keys
from stl_runner.storage import ArtifactStore

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

# Reference categorical palette (slot 1 = meta-model, slot 2 = baseline); text stays in neutral ink.
C_MODEL, C_BASE, C_NEUTRAL, INK, INK_2 = "#2a78d6", "#eb6834", "#8a8984", "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": INK_2, "axes.labelcolor": INK, "text.color": INK,
                     "xtick.color": INK_2, "ytick.color": INK_2, "figure.dpi": 100})

store = ArtifactStore(STORE_ROOT)

lodo, lodo_error = None, None
try:
    if store.exists(keys.meta_lodo()):
        lodo = store.read_json(keys.meta_lodo())
    else:
        lodo_error = f"{keys.meta_lodo()} not found -- run `experiments/run.py meta-eval` first"
except Exception as e:  # noqa: BLE001 -- report, don't crash the notebook
    lodo_error = f"could not read {keys.meta_lodo()}: {e!r}"

artifact, artifact_error = None, None
try:
    artifact = meta.load_artifact(store)
    if artifact is None:
        artifact_error = f"{keys.meta_predictor()} not found -- run `meta-eval --train`"
except Exception as e:  # noqa: BLE001
    artifact_error = f"could not load {keys.meta_predictor()}: {e!r}"

INSUFFICIENT = lodo is None or bool(lodo.get("insufficient_data"))
WHY_INSUFFICIENT = lodo_error if lodo is None else lodo.get("reason")

print("lodo.json:", "loaded" if lodo else lodo_error)
print("predictor:", "loaded" if artifact else artifact_error)
if INSUFFICIENT:
    print(f"\nINSUFFICIENT DATA for LODO metrics: {WHY_INSUFFICIENT}")
    print("Sections 3-5 only explain this; the dataset table and label distribution are still shown.")

## 1. Protocol summary

* **Datasets:** OpenML-CC18 (suite 99) classification tasks with 500 ≤ instances ≤ 20,000, ≤ 60 features,
  ≤ 20 classes and enough minority-class rows for a 60/20/20 split; first 25 by OpenML id, frozen in
  `experiments/conf/meta/datasets.yaml` with the exclusions and their reasons.
* **Generators:** `independent_marginals, gaussian_copula, bayesian_network, ctgan, tvae, mst_eps1`
  (+ `train_copy` as a sanity ceiling, never a label), budget profile `equal50` (50 epochs, 7,200 s cap),
  `max_rows=5000`, seeds 0,1,2; datasets use `openml_generic` (no public bounds), so MST spends 0.2·ε on
  privately inferred bounds for continuous columns.
* **Label:** `recommend(aggregate(reports), "balanced")` -- the highest balanced score among gate-passing
  candidates, seeds aggregated; `none` if no candidate passes (kept, reported, excluded from training and
  accuracy). Datasets with missing/failed meta-features, no run or an incomplete run are excluded.
* **Meta-features:** pymfe `general`, `statistical`, `info-theory` (`mean`, `sd`) on the seed-0 TRAIN split.
* **Evaluation:** LODO; per fold the medians used for imputation, the
  `RandomForestClassifier(n_estimators=300, random_state=0, class_weight="balanced")` and the majority
  baseline are fitted on the training folds only. Metrics: accuracy, balanced accuracy, mean regret (below),
  and `n_gate_failing` (picks that failed the gate on the held-out dataset).

In [ ]:
with open(DATASETS_YAML, encoding="utf-8") as f:
    suite = yaml.safe_load(f) or {}
selected = suite.get("datasets") or []
sel_excluded = suite.get("excluded") or []
print(f"selected datasets: {len(selected)}; excluded at selection: {len(sel_excluded)}")
if lodo is not None:
    print("model:", lodo.get("model_params"))
    print("regret definition:", lodo.get("regret_definition"))
pd.DataFrame(sel_excluded, columns=["id", "name", "reason"]).rename(
    columns={"reason": "selection exclusion reason"})

## 2. Datasets: labelled / `none` / excluded

One row per frozen dataset. The meta-dataset is rebuilt read-only with the same function `meta-eval` uses
(`stl_meta.meta.build_meta_dataset`) and checked against `lodo.json`. The `mst` column comes from
`mst_bounds.txt`: `failed` means the private-bounds fallback could not run (too little ε per continuous
column for the dataset size), so `mst_eps1` was **not a candidate** there and labels are conditional on that.

In [ ]:
def parse_mst_bounds(path):
    # Lines: '<run> <job> bounds=<path> ...' or '<run> <job> bounds=? state=failed (<reason>)'.
    rows = []
    if not path.exists():
        return pd.DataFrame(columns=["run", "job", "bounds", "failed", "detail"])
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip() or line.startswith("#"):
            continue
        parts = line.split(maxsplit=2)
        if len(parts) < 3:
            continue
        rest = parts[2]
        m = re.match(r"bounds=(\S+)", rest)
        rows.append({"run": parts[0], "job": parts[1], "bounds": m.group(1) if m else "?",
                     "failed": "state=failed" in rest, "detail": rest})
    return pd.DataFrame(rows)


mst = parse_mst_bounds(MST_BOUNDS)


def mst_status(did):
    jobs = mst[mst["run"] == f"{RUN_PREFIX}{did}"] if len(mst) else mst
    if not len(jobs):
        return "n/a"
    n_fail = int(jobs["failed"].sum())
    paths = ",".join(sorted(set(jobs.loc[~jobs["failed"], "bounds"])))
    if n_fail == len(jobs):
        return f"failed ({n_fail}/{len(jobs)} seeds)"
    return f"{paths}" + (f", {n_fail}/{len(jobs)} failed" if n_fail else "")


meta_df, meta_df_error = None, None
try:
    meta_df = meta.build_meta_dataset(store, DATASETS_YAML, run_prefix=RUN_PREFIX)
except Exception as e:  # noqa: BLE001
    meta_df_error = repr(e)
    print("could not rebuild the meta-dataset:", meta_df_error)

excluded = (meta_df.attrs.get("excluded", []) if meta_df is not None
            else (lodo or {}).get("excluded", []))
excl_reason = {e["dataset_id"]: e["reason"] for e in excluded}
rows_by_id = ({r["dataset_id"]: r for _, r in meta_df.iterrows()}
              if meta_df is not None and len(meta_df) else {})
none_ids = set((lodo or {}).get("none_datasets", []))

table = []
for item in selected:
    did = f"cc18_{item['id']}"
    r = rows_by_id.get(did)
    if r is not None:
        status = "none" if r["label"] == meta.NONE_LABEL else "labelled"
        label = r["label"]
        n_pass = sum(bool(r[c]) for c in r.index
                     if str(c).startswith(meta.PASS_PREFIX) and not pd.isna(r[c]))
        reason = "no generator passed the privacy gate" if status == "none" else ""
    elif did in excl_reason:
        status, label, n_pass, reason = "excluded", "", None, excl_reason[did]
    else:
        status = "none" if did in none_ids else "unknown"
        label, n_pass, reason = "", None, ""
    table.append({"dataset_id": did, "name": item.get("name"), "n_instances": item.get("n_instances"),
                  "n_features": item.get("n_features"), "n_classes": item.get("n_classes"),
                  "status": status, "label": label, "n_gate_passing": n_pass,
                  "mst": mst_status(did), "reason": reason})
datasets_table = pd.DataFrame(table)
datasets_table["n_gate_passing"] = datasets_table["n_gate_passing"].astype("Int64")

counts = datasets_table["status"].value_counts().to_dict()
print("status counts:", counts)
if lodo is not None and meta_df is not None:
    same = (counts.get("labelled", 0) == lodo.get("n_labelled")
            and counts.get("none", 0) == lodo.get("n_none"))
    print("consistent with lodo.json:", same,
          "" if same else "-- lodo.json is stale; re-run `meta-eval`")
mst_lost = datasets_table[datasets_table["mst"].str.startswith("failed")]
print(f"datasets where mst_eps1 failed every seed (private-bounds fallback): {len(mst_lost)}"
      + (f" -> {', '.join(mst_lost['dataset_id'])}" if len(mst_lost) else ""))
if not MST_BOUNDS.exists():
    print("(mst_bounds.txt not found -- MST losses unknown)")
datasets_table

## 3. LODO accuracy, balanced accuracy and mean regret vs the majority baseline

Baseline = the most frequent winner of the training folds (ties alphabetical). Regret (lower is better) is
the balanced score lost against the best gate-passing generator on the held-out dataset; a gate-failing pick
costs as much as the worst gate-passing pick, which is 0 when only one generator passed -- hence
`n_gate_failing` is reported separately.

In [ ]:
if INSUFFICIENT:
    print(f"Insufficient data -- no LODO metrics: {WHY_INSUFFICIENT}")
    print("LODO needs >= 3 labelled datasets with >= 2 distinct winners; run the full meta-benchmark.")
else:
    metric_rows = [("Accuracy", "accuracy", "baseline_accuracy"),
                   ("Balanced accuracy", "balanced_accuracy", "baseline_balanced_accuracy"),
                   ("Mean regret (lower is better)", "mean_regret", "baseline_mean_regret"),
                   ("Gate-failing picks (n_gate_failing)", "n_gate_failing", "baseline_n_gate_failing")]
    metrics_df = pd.DataFrame([{"metric": name, "meta-model": lodo.get(m), "baseline": lodo.get(b)}
                               for name, m, b in metric_rows])
    print(f"labelled datasets in LODO: {lodo['n_labelled']} (none: {lodo['n_none']}, "
          f"excluded: {len(lodo.get('excluded', []))}); meta-features: {len(lodo['features'])}")
    display(metrics_df)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    panels = [("Accuracy", "accuracy", "baseline_accuracy", (0, 1.05)),
              ("Balanced accuracy", "balanced_accuracy", "baseline_balanced_accuracy", (0, 1.05)),
              ("Mean regret (lower is better)", "mean_regret", "baseline_mean_regret", None)]
    for ax, (title, m, b, ylim) in zip(axes, panels, strict=True):
        vals = [lodo.get(m), lodo.get(b)]
        shown = [0.0 if v is None else float(v) for v in vals]
        bars = ax.bar(["meta-model", "baseline"], shown, color=[C_MODEL, C_BASE], width=0.55,
                      edgecolor="white", linewidth=2)
        for bar, v in zip(bars, vals, strict=True):
            ax.annotate("n/a" if v is None else f"{v:.3f}",
                        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        xytext=(0, 3), textcoords="offset points", ha="center", color=INK)
        if ylim:
            ax.set_ylim(*ylim)
        else:
            ax.set_ylim(0, max(shown + [1e-3]) * 1.25)
        ax.set_title(title, color=INK)
        ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
        ax.set_axisbelow(True)
    fig.suptitle(f"LODO on {lodo['n_labelled']} labelled datasets -- gate-failing picks: "
                 f"meta-model {lodo.get('n_gate_failing')}, baseline {lodo.get('baseline_n_gate_failing')}",
                 color=INK)
    plt.tight_layout()
    plt.show()

## 4. Per-dataset LODO predictions

In [ ]:
preds = (lodo or {}).get("predictions") or []
if not preds:
    print(f"No per-dataset predictions: {WHY_INSUFFICIENT or 'empty prediction list'}")
else:
    preds_df = pd.DataFrame(preds)
    preds_df["correct"] = preds_df["true"] == preds_df["predicted"]
    preds_df["baseline_correct"] = preds_df["true"] == preds_df["baseline"]
    if meta_df is not None and len(meta_df):
        names = meta_df.set_index("dataset_id")["name"]
        preds_df.insert(1, "name", preds_df["dataset_id"].map(names))
    display(preds_df[[c for c in ["dataset_id", "name", "true", "predicted", "baseline", "correct",
                                  "baseline_correct", "regret", "baseline_regret",
                                  "predicted_passed_gate", "baseline_passed_gate"]
                      if c in preds_df.columns]])

## 5. Confusion matrix (true vs predicted winner, LODO)

In [ ]:
if not preds:
    print(f"No confusion matrix: {WHY_INSUFFICIENT or 'no predictions'}")
else:
    labels = sorted(set(preds_df["true"]) | set(preds_df["predicted"]))
    cm = pd.crosstab(pd.Categorical(preds_df["true"], categories=labels),
                     pd.Categorical(preds_df["predicted"], categories=labels),
                     rownames=["true"], colnames=["predicted"], dropna=False)
    display(cm)
    fig, ax = plt.subplots(figsize=(1.0 * len(labels) + 3, 0.8 * len(labels) + 2.5))
    im = ax.imshow(cm.values, cmap="Blues", vmin=0)
    ax.set_xticks(range(len(labels)), labels, rotation=35, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true winner")
    vmax = cm.values.max() or 1
    for i in range(len(labels)):
        for j in range(len(labels)):
            v = cm.values[i, j]
            ax.text(j, i, str(v), ha="center", va="center",
                    color="white" if v > vmax / 2 else INK)
    ax.spines[:].set_visible(False)
    ax.set_title("LODO confusion matrix")
    plt.tight_layout()
    plt.show()

## 6. Feature importances (final predictor)

Impurity-based importances of the final RandomForest (`meta-eval --train`, all labelled datasets).
With ~25 datasets they are unstable and only descriptive -- not evidence of a causal link.

In [ ]:
if artifact is None:
    print(f"No final predictor: {artifact_error}")
else:
    md_ = artifact.get("metadata", {})
    n_train = md_.get("n_datasets")
    print(f"predictor: trained_at={md_.get('trained_at')}, n_datasets={n_train}, "
          f"labels={artifact.get('labels')}")
    if lodo is not None and n_train != lodo.get("n_labelled"):
        print(f"WARNING: predictor trained on {n_train} datasets but lodo.json has "
              f"{lodo.get('n_labelled')} labelled -- predictor and LODO are from different runs "
              "(re-run `meta-eval --train`).")
    model = artifact.get("model")
    if n_train is None or n_train < meta.MIN_LABELLED or not hasattr(model, "feature_importances_"):
        print(f"Insufficient data: a predictor trained on {n_train} dataset(s) "
              f"(< {meta.MIN_LABELLED}) has meaningless importances -- plot skipped. "
              "(Likely the 2-dataset smoke predictor; delete it before the full run.)")
    else:
        imp = (pd.Series(model.feature_importances_, index=artifact["features"])
               .sort_values(ascending=False).head(TOP_N))
        display(imp.rename("importance").to_frame())
        fig, ax = plt.subplots(figsize=(8, 0.32 * len(imp) + 1.2))
        ax.barh(imp.index[::-1], imp.values[::-1], color=C_MODEL, height=0.6)
        ax.set_xlabel("mean decrease in impurity")
        ax.set_title(f"Top {len(imp)} meta-features (final predictor, {n_train} datasets)")
        ax.grid(axis="x", color="#e4e3df", linewidth=0.8)
        ax.set_axisbelow(True)
        plt.tight_layout()
        plt.show()

## 7. Label distribution

Winner labels over the meta-dataset (`none` = no gate-passing generator). A dominant class makes the
majority baseline hard to beat; generators absent here never won (or, for `mst_eps1`, may have failed).

In [ ]:
label_source = None
if meta_df is not None and len(meta_df):
    label_source = meta_df["label"].astype(str)
elif preds:  # meta-dataset not rebuildable here (e.g. another store): fall back to lodo.json
    label_source = pd.Series([p["true"] for p in preds]
                             + [meta.NONE_LABEL] * int(lodo.get("n_none") or 0))
    print("(labels from lodo.json predictions; runs not found in this store)")
if label_source is None:
    print("No labelled or none datasets yet" + (f" ({meta_df_error})" if meta_df_error else "")
          + " -- run the meta-benchmark.")
else:
    dist = label_source.value_counts().sort_values(ascending=False)
    display(dist.rename("n_datasets").to_frame())
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(dist) + 1.2))
    colors = [C_NEUTRAL if lab == meta.NONE_LABEL else C_MODEL for lab in dist.index[::-1]]
    bars = ax.barh(dist.index[::-1], dist.values[::-1], color=colors, height=0.6)
    for bar, v in zip(bars, dist.values[::-1], strict=True):
        ax.annotate(str(v), (bar.get_width(), bar.get_y() + bar.get_height() / 2),
                    xytext=(3, 0), textcoords="offset points", va="center", color=INK)
    ax.set_xlabel("datasets")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.set_title(f"Winner labels ({len(label_source)} datasets; grey = none)")
    plt.tight_layout()
    plt.show()
    if len(label_source) < meta.MIN_LABELLED:
        print("(descriptive only -- too few datasets for any conclusion)")

## 8. Discussion and threats to validity

### Reading the results
- **Regret vs accuracy:** mean regret measures how much balanced score is lost by following the
  suggestion instead of the ex-post best gate-passing generator. A meta-model with lower regret than the
  baseline can be useful even when accuracy is modest (near-ties between generators make "wrong" picks
  cheap); `n_gate_failing` shows how often a suggestion would have been blocked by the privacy gate.
- **Negative results are valid:** if the meta-model does not beat the majority baseline, that is the RQ3
  answer for this suite -- the dataset profile alone does not identify the winner at this sample size.

### Threats to validity
- **Small meta-dataset:** at most 25 datasets (fewer after `none`/exclusions); LODO estimates have high
  variance and one dataset moves accuracy by ~4 points. No significance test is claimed.
- **MST fallback bias:** without public bounds, MST infers bounds privately with 0.2·ε; on small numeric
  datasets this fails, so `mst_eps1` cannot be a candidate there (see the `mst` column in section 2).
  Labels are conditional on that, and MST wins are biased towards categorical / larger datasets.
- **DOMIAS not applicable on all-categorical datasets** (e.g. ids 3, 11, 46, 50, 469): the gate's MIA
  check there relies on the distance-based MIA only.
- **Fixed budget:** `equal50` on CPU; deep generators might rank differently with library-default or
  larger budgets. Labels also depend on the `balanced` preset weights.